# Institutional Quant Research Engine V2.1

Thin reproducible interface. **All core logic lives in `src/quant_research/`**
- this notebook only orchestrates: configuration -> data -> PIT validation ->
features -> walk-forward baseline -> robustness -> ablation -> experiment
record -> promotion status.

Research contract: never tune on test; validation is the selection layer;
test is evaluation-only; synthetic results are NOT market evidence.

## 1. Configuration

In [1]:
from quant_research.config import load_config
from quant_research import __version__

CONFIG_PATH = "configs/baseline.yaml"   # switch to configs/real_spy.yaml for real data
cfg = load_config(CONFIG_PATH)
print(f"engine {__version__} | config fingerprint: {cfg.fingerprint()}")
cfg.to_dict()

engine 2.1.3 | config fingerprint: fa244c8424e0659c


{'data': {'mode': 'synthetic',
  'assets': ['SPY', 'QQQ', 'AAPL', 'MSFT', 'JPM', 'GLD', 'TLT'],
  'target': 'SPY',
  'start': '2012-01-01',
  'end': '2026-01-01',
  'frequency': '1d',
  'csv_path': None,
  'raw_snapshot_dir': 'data/raw_snapshots'},
 'evaluation': {'train_window': 1260,
  'validation_window': 252,
  'test_window': 252,
  'step_bars': 252,
  'purge_bars': 5,
  'embargo_bars': 5,
  'expanding': True},
 'execution': {'fee_bps': 5.0,
  'slippage_bps': 1.0,
  'signal_delay_bars': 0,
  'target_vol': 0.1,
  'max_position': 1.0},
 'model': {'type': 'logistic',
  'random_seed': 42,
  'parameters': {'C': 1.0, 'max_iter': 1000}},
 'research': {'max_trials': 48,
  'bootstrap_samples': 500,
  'placebo_runs': 20,
  'threshold_candidates': [0.5, 0.52, 0.54, 0.56, 0.58, 0.6, 0.62, 0.64, 0.66],
  'hold_candidates': [1, 2, 3, 5]},
 'promotion': {'min_median_oos_sharpe': 0.0,
  'min_mean_oos_sharpe': 0.0,
  'max_oos_dd': -0.5,
  'cost_stress_fee_bps': 10.0,
  'delay_stress_bars': 1,
  'mi

## 2. Data load & validation

In [2]:
from quant_research.data.loaders import load_market_data, to_panels
from quant_research.data.validation import validate_ohlcv, missing_data_report
from quant_research.data.snapshots import save_snapshot

ohlcv, data_meta = load_market_data(cfg.data)
ohlcv = validate_ohlcv(ohlcv)          # fails loudly on schema violations
snapshot = save_snapshot(ohlcv, cfg.data.raw_snapshot_dir, name=f"{cfg.data.mode}_ohlcv")
missing = missing_data_report(ohlcv)
print(f"dataset_hash={snapshot['dataset_hash']} rows={len(ohlcv)}")
missing

dataset_hash=5da40b7b90035c78 rows=24640


,symbol,n_obs,first,last,n_expected_sessions,n_missing_sessions,n_observed_closures,n_weekend_bars,n_holiday_bars,n_zero_volume
0,AAPL,3520,2012-01-03 00:00:00+00:00,2025-12-31 00:00:00+00:00,3520,0,0,0,0,0
1,GLD,3520,2012-01-03 00:00:00+00:00,2025-12-31 00:00:00+00:00,3520,0,0,0,0,0
2,JPM,3520,2012-01-03 00:00:00+00:00,2025-12-31 00:00:00+00:00,3520,0,0,0,0,0
3,MSFT,3520,2012-01-03 00:00:00+00:00,2025-12-31 00:00:00+00:00,3520,0,0,0,0,0
4,QQQ,3520,2012-01-03 00:00:00+00:00,2025-12-31 00:00:00+00:00,3520,0,0,0,0,0
5,SPY,3520,2012-01-03 00:00:00+00:00,2025-12-31 00:00:00+00:00,3520,0,0,0,0,0
6,TLT,3520,2012-01-03 00:00:00+00:00,2025-12-31 00:00:00+00:00,3520,0,0,0,0,0


## 3. Point-in-time event validation

In [3]:
from quant_research.run import generate_synthetic_events
from quant_research.features.point_in_time import validate_events

# Synthetic mode exercises the PIT layer with clearly-labelled synthetic
# events; real information feeds enter through the same validated schema.
events = None
if cfg.data.mode == "synthetic":
    events = validate_events(generate_synthetic_events(
        ohlcv[ohlcv.symbol == cfg.data.target]["timestamp"], cfg.data.target))
    print("PIT event schema validated (synthetic events; NOT market evidence)")

PIT event schema validated (synthetic events; NOT market evidence)


## 4. Feature construction + leakage check

In [4]:
from quant_research.data.loaders import to_price_panels
from quant_research.features.price_volume import build_price_volume_features, build_signal_extensions
from quant_research.features.information import build_information_features
from quant_research.features.leakage import feature_leakage_report
from quant_research.features.registry import registry_hash

# E22: bind the CURRENT pipeline feature set (price/volume pv-2.1.0 +
# signal extensions pv-2.2.0 + information).  Reconstructing a stale
# price/volume-only subset here would evaluate a different experiment
# than run_research_pipeline records.
close, volume = to_panels(ohlcv)
open_, high, low, _close, _vol = to_price_panels(ohlcv)
price_feats = build_price_volume_features(close, volume, cfg.data.target)
signal_ext = build_signal_extensions(open_, high, low, _close, _vol, cfg.data.target)
price_feats = price_feats.join(signal_ext, how="left")
if events is not None:
    info_feats = build_information_features(close.index, events, cfg.data.target)
    features = price_feats.join(info_feats, how="left")
else:
    features = price_feats
leakage = feature_leakage_report(close, volume, cfg.data.target,
                                 info_events=events,
                                 open_=open_, high=high, low=low)
print(f"features={features.shape} feature_version={registry_hash(list(features.columns))}")
print("leakage check passed:", leakage["passed"])


features=(3520, 19) feature_version=79da8435f58972e7
leakage check passed: True


## 5. Walk-forward baseline (TRAIN -> VAL -> PURGE/EMBARGO -> TEST)

In [5]:
import numpy as np
from quant_research.evaluation.walk_forward import LockedTestProtocol
from quant_research.experiments.registry import TrialCounter
from quant_research.strategies.baseline import run_walk_forward, summarize_experiment

y = (close[cfg.data.target].shift(-1) > close[cfg.data.target]).astype("float")
y[close[cfg.data.target].shift(-1).isna()] = np.nan
fwd = close[cfg.data.target].shift(-1) / close[cfg.data.target] - 1.0

# E23: the notebook binds the SAME trial it evaluates (per-trial binding).
# The counter is read but never mutated by evaluation, so comparing a
different experiment's record can never gate this run.
locked_test = LockedTestProtocol()          # freeze the test layout
counter = TrialCounter("artifacts/trial_counter.json")
baseline = run_walk_forward(features, y, fwd, cfg,
                            locked_test=locked_test, trial_counter=counter)
summary = summarize_experiment(baseline)
print(f"global trials: {counter.count}")
print(f"trials_this_experiment: {baseline.trials_this_experiment}")
baseline.folds


global trials: 576


,fold_id,train_start,train_end,val_start,val_end,test_start,test_end,purge_bars,embargo_bars,n_train,...,oos_sortino,oos_cagr,oos_max_dd,oos_trades,oos_turnover,oos_gross_return,oos_net_return,oos_auc,oos_brier,n_trials_this_fold
0,1,2012-01-03,2017-01-04,2017-01-12,2018-01-11,2018-01-22,2019-01-22,5,5,1260,...,1.056592,0.061793,-0.093990,251,6.483297,0.065936,0.061793,0.494291,0.311305,9
1,2,2012-01-03,2018-01-04,2018-01-12,2019-01-14,2019-01-23,2020-01-22,5,5,1512,...,NaN,0.000000,0.000000,0,0.000000,0.000000,0.000000,0.516965,0.262594,9
2,3,2012-01-03,2019-01-07,2019-01-15,2020-01-14,2020-01-23,2021-01-21,5,5,1764,...,0.058960,0.004241,-0.051520,47,17.633964,0.014926,0.004241,0.514326,0.258723,9
3,4,2012-01-03,2020-01-07,2020-01-15,2021-01-13,2021-01-22,2022-01-20,5,5,2016,...,-0.286198,-0.004730,-0.004730,3,1.260185,-0.003977,-0.004730,0.535941,0.261158,9
4,5,2012-01-03,2021-01-06,2021-01-14,2022-01-12,2022-01-21,2023-01-23,5,5,2268,...,-0.505462,-0.026272,-0.071155,116,44.366013,-0.000009,-0.026272,0.490800,0.249852,9
5,6,2012-01-03,2022-01-05,2022-01-13,2023-01-13,2023-01-24,2024-01-24,5,5,2520,...,0.329605,0.015718,-0.051136,144,44.505845,0.043214,0.015718,0.505971,0.249669,9
6,7,2012-01-03,2023-01-06,2023-01-17,2024-01-17,2024-01-25,2025-01-27,5,5,2772,...,NaN,0.000000,0.000000,0,0.000000,0.000000,0.000000,0.494038,0.245406,9


## 6. Robustness battery

In [6]:
from quant_research.evaluation.robustness import (
    cost_stress, slippage_stress, delay_stress)

# Stresses run on the EXACT fold-level OOS execution path: same folds,
# selected features, parameters, per-fold thresholds, preprocessing,
# sizing.  Only the stressed variable changes.
cost_table = cost_stress(features, y, fwd, cfg, baseline, locked_test)
slip_table = slippage_stress(features, y, fwd, cfg, baseline, locked_test)
delay_table = delay_stress(features, y, fwd, cfg, baseline, locked_test)
display(cost_table); display(slip_table); display(delay_table)


,sharpe,gross_sharpe,net_return,gross_return,max_dd,annual_turnover,fee_cost,slippage_cost,cost_drag,fee_bps,slippage_bps,delay_bars
0,0.302808,0.332628,0.111328,0.124097,-0.093101,16.321329,0.000000,0.011425,0.012770,0.0,1.0,0
1,0.228234,0.332628,0.080032,0.124097,-0.093546,16.321329,0.028562,0.011425,0.044066,2.5,1.0,0
2,0.153657,0.332628,0.049612,0.124097,-0.093990,16.321329,0.057125,0.011425,0.074485,5.0,1.0,0
3,0.004661,0.332628,-0.008693,0.124097,-0.094879,16.321329,0.114249,0.011425,0.132790,10.0,1.0,0
4,-0.291546,0.332628,-0.115814,0.124097,-0.177471,16.321329,0.228499,0.011425,0.239911,20.0,1.0,0


,sharpe,gross_sharpe,net_return,gross_return,max_dd,annual_turnover,fee_cost,slippage_cost,cost_drag,fee_bps,slippage_bps,delay_bars
0,0.183485,0.332628,0.061677,0.124097,-0.093812,16.321329,0.057125,0.000000,0.062421,5.0,0.0,0
1,0.153657,0.332628,0.049612,0.124097,-0.093990,16.321329,0.057125,0.011425,0.074485,5.0,1.0,0
2,0.123834,0.332628,0.037685,0.124097,-0.094168,16.321329,0.057125,0.022850,0.086413,5.0,2.0,0
3,0.034432,0.332628,0.002704,0.124097,-0.094701,16.321329,0.057125,0.057125,0.121393,5.0,5.0,0
4,-0.114200,0.332628,-0.053009,0.124097,-0.124246,16.321329,0.057125,0.114249,0.177106,5.0,10.0,0


,sharpe,gross_sharpe,net_return,gross_return,max_dd,annual_turnover,fee_cost,slippage_cost,cost_drag,fee_bps,slippage_bps,delay_bars
0,0.153657,0.332628,0.049612,0.124097,-0.093990,16.321329,0.057125,0.011425,0.074485,5.0,1.0,0
1,0.385381,0.558003,0.150129,0.230786,-0.111202,16.145420,0.056509,0.011302,0.080657,5.0,1.0,1
2,-0.000241,0.167506,-0.011457,0.057180,-0.093990,15.984707,0.055946,0.011189,0.068637,5.0,1.0,2
3,-0.058352,0.119042,-0.031643,0.035327,-0.124320,15.915488,0.055704,0.011141,0.066970,5.0,1.0,3


## 7. Information-source ablation + empirical null

In [7]:
import pandas as pd
from quant_research.evaluation.placebo import run_placebo_null, placebo_statistics

def _summarize(feats):
    return summarize_experiment(run_walk_forward(feats, y, fwd, cfg, locked_test=locked_test))

ablation = [{"source": "price_volume",
             "mean_oos_sharpe": _summarize(price_feats)["mean_oos_sharpe"]}]
if events is not None:
    ablation.append({"source": "price_plus_information",
                     "mean_oos_sharpe": summary["mean_oos_sharpe"]})
null = run_placebo_null(features, y, fwd, lambda X, yy, ff: _summarize(X),
                        n_runs=cfg.research.placebo_runs, seed=cfg.model.random_seed)
placebo = placebo_statistics(summary["mean_oos_sharpe"], null)
display(pd.DataFrame(ablation))
print("placebo:", placebo)

,source,mean_oos_sharpe
0,price_volume,-0.452831
1,price_plus_information,-0.193747


placebo: {'percentile': 0.35, 'adjusted_p': 0.6666666666666666, 'null_mean': -0.08581915630279846, 'null_median': -0.06311399526415425, 'null_std': 0.35310994947154317, 'null_p95': 0.45002235196194174, 'n_runs': 20, 'observed': -0.19374733030757507}


## 8. Experiment record + promotion status

In [8]:
from quant_research.run import run_research_pipeline

# Full pipeline execution (same config): registers the immutable experiment
# record, writes artifacts, and applies the promotion gates.
report = run_research_pipeline(cfg, cfg.output_dir)
record = report["experiment_record"]
print("experiment_id:", record["experiment_id"])
print("evidence_status:", record["evidence_status"])
print("promotion_state:", record["promotion_state"])
print("failed gates:", record["failed_gates"] or "none")
report["leaderboard"]

experiment_id: 20260907T194304Z_4dec63bf3678b566
evidence_status: SYNTHETIC_OFFLINE
promotion_state: RESEARCH_ONLY
failed gates: ['mean_oos_sharpe_positive', 'cost_stress_survives', 'delay_stress_survives', 'placebo_separates']


,experiment_id,timestamp_utc,strategy,data_mode,target,universe,trials,n_trials_global,search_class,mean_oos_sharpe,...,bootstrap_lo,bootstrap_hi,survives_cost_stress,survives_delay_stress,placebo_percentile,robustness_score,dataset_version,feature_version,promotion_state,failed_gates
0,20260907T053806Z_ea1118bc409f7e12,20260907T053806Z,walk_forward_baseline,synthetic,SPY,"[SPY, QQQ, AAPL, MSFT, JPM, GLD, TLT]",72,72,large_scale_search,-0.390356,...,-1.110681,0.369435,False,False,0.45,0.0,70fc00cb0f0c0e09,79da8435f58972e7,RESEARCH_ONLY,median_oos_sharpe_positive;mean_oos_sharpe_pos...
1,20260907T145014Z_d838719480849dff,20260907T145014Z,walk_forward_baseline,synthetic,SPY,"[SPY, QQQ, AAPL, MSFT, JPM, GLD, TLT]",261,261,large_scale_search,0.027219,...,-0.479894,0.756553,False,False,0.65,0.0,5da40b7b90035c78,79da8435f58972e7,RESEARCH_ONLY,median_oos_sharpe_positive;cost_stress_survive...
2,20260907T164935Z_06c4faf68de6c642,20260907T164935Z,walk_forward_baseline,synthetic,SPY,"[SPY, QQQ, AAPL, MSFT, JPM, GLD, TLT]",387,387,large_scale_search,0.027219,...,-0.479894,0.756553,False,False,0.65,0.0,5da40b7b90035c78,79da8435f58972e7,RESEARCH_ONLY,median_oos_sharpe_positive;cost_stress_survive...
3,20260907T174244Z_b217114485f7b656,20260907T174244Z,walk_forward_baseline,synthetic,SPY,"[SPY, QQQ, AAPL, MSFT, JPM, GLD, TLT]",513,513,large_scale_search,0.027219,...,-0.479894,0.756553,False,False,0.65,0.0,5da40b7b90035c78,79da8435f58972e7,RESEARCH_ONLY,median_oos_sharpe_positive;cost_stress_survive...
4,20260907T194304Z_4dec63bf3678b566,20260907T194304Z,walk_forward_baseline,synthetic,SPY,"[SPY, QQQ, AAPL, MSFT, JPM, GLD, TLT]",639,639,large_scale_search,-0.193747,...,-0.430309,0.845243,False,False,0.35,0.0,5da40b7b90035c78,79da8435f58972e7,RESEARCH_ONLY,mean_oos_sharpe_positive;cost_stress_survives;...
